# Pallet OBB trainer (easy)

This notebook trains the YOLOX oriented-box pallet detector. You do **not** need to know PyTorch.

## What to do (3 clicks)

1. In the **panel** cell, pick **STAGE** and **INIT**, click **Apply**, then **Run All**.
2. Wait for training. Look at the pictures.
3. If boxes find pallets but stay level, set STAGE = `angle`, INIT = `fresh`, Apply, Run All again.
4. Set STAGE = `export`, Apply, Run All — download `export/yolox_s_obb.pt`.

| STAGE | INIT = **fresh** | INIT = **continue** |
|---|---|---|
| `detect` | Start from public YOLOX-S (first train ever) | Finetune from a checkpoint you uploaded, or `01_detect_best.pth` |
| `angle` | Start tilt training from `01_detect_best.pth` | Resume tilt from `02_angle_best.pth` if it exists |
| `export` | Skip train; pack the latest file | Same |

Weights are always copied to **`pallet_weights/`** with stable names. You never type `best_ckpt.pth`.

**Kaggle:** GPU + Internet on. Dataset = Input *or* `DATASET_URL`. Need `train/images`, `train/labels`, `valid/images`, `valid/labels`.

**New session:** `/kaggle/working` is empty. Upload `01_detect_best.pth` as Input and set INIT = `continue`.


## 0. Control panel — pick STAGE / INIT, then Apply


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Defaults (the panel below overwrites these when you click Apply)
STAGE = "detect"          # "detect" | "angle" | "export"
INIT = "fresh"            # "fresh" | "continue"
START_WEIGHTS = None      # Path("/kaggle/input/my-weights/best_ckpt-3.pth") or None
KAGGLE_DATA_DIR = None
DATASET_URL = ""
DATASET_ZIP = None
MAX_EPOCH_OVERRIDE = 0    # 0 = default (detect 100 / angle 30). Use 40 for ~200 images.
MODEL = "s"
GIT_BRANCH = "main"
CLEAN_START = True
FINETUNE_FREEZE = "full"

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = False
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    pass

ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("/content") if IS_COLAB else Path.cwd()
REPO_URL = "https://github.com/TANISHQBEDI/YOLOX.git"
YOLOX_DIR = ROOT / "YOLOX"
DATASET_DIR = ROOT / "dataset"
BANK = ROOT / "pallet_weights"
EXPORT = ROOT / "export"
BANK.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)
DETECT_BEST = BANK / "01_detect_best.pth"
ANGLE_BEST = BANK / "02_angle_best.pth"
COCO_WEIGHTS = YOLOX_DIR / "weights" / f"yolox_{MODEL}.pth"
OUT_DIR = YOLOX_DIR / "YOLOX_outputs" / f"yolox_{MODEL}_pallets_kaggle"
_CKPT = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0"
MODEL_ZOO = {
    "s": dict(depth=0.33, width=0.50, depthwise=False, img_size=416, batch=8, mixup=True, mosaic_prob=1.0, ckpt=f"{_CKPT}/yolox_s.pth"),
}
CFG = MODEL_ZOO[MODEL]
WEIGHTS_URL = CFG["ckpt"]
KAGGLE_EXP_REL = Path(f"exps/example/custom/yolox_{MODEL}_pallets_kaggle.py")
NUM_CLASSES, IMG_SIZE, BATCH, DEVICES = 1, CFG["img_size"], CFG["batch"], 1
FP16, DATA_NUM_WORKERS, EVAL_INTERVAL = True, 2, 5
EVAL_CONF, EVAL_NMS, FINETUNE_LR_SCALE = 0.25, 0.45, 0.1


def _sync_stage_knobs():
    global MAX_EPOCH, REG_WEIGHT, ANGLE_WEIGHT, NO_AUG_EPOCHS, WIPE_OUTPUTS
    if STAGE == "detect":
        MAX_EPOCH = MAX_EPOCH_OVERRIDE or 100
        REG_WEIGHT, ANGLE_WEIGHT = 5.0, 0.5
        NO_AUG_EPOCHS = 15 if MAX_EPOCH >= 80 else 5
        WIPE_OUTPUTS = INIT == "fresh"
    elif STAGE == "angle":
        MAX_EPOCH = MAX_EPOCH_OVERRIDE or 30
        REG_WEIGHT, ANGLE_WEIGHT = 1.0, 5.0
        NO_AUG_EPOCHS, WIPE_OUTPUTS = 5, False
    else:
        MAX_EPOCH, REG_WEIGHT, ANGLE_WEIGHT = 0, 5.0, 0.5
        NO_AUG_EPOCHS, WIPE_OUTPUTS = 5, False


def print_plan():
    _sync_stage_knobs()
    print("================================================")
    print(f"STAGE={STAGE}   INIT={INIT}   epochs={MAX_EPOCH}")
    print(f"box_loss x{REG_WEIGHT}   angle_loss x{ANGLE_WEIGHT}")
    print(f"bank   {BANK}")
    print(f"export {EXPORT}")
    print("detect →", DETECT_BEST.name)
    print("angle  →", ANGLE_BEST.name)
    print("================================================")


_sync_stage_knobs()
print_plan()

try:
    import ipywidgets as W
    from IPython.display import display

    stage_w = W.Dropdown(options=["detect", "angle", "export"], value=STAGE, description="STAGE")
    init_w = W.Dropdown(options=["fresh", "continue"], value=INIT, description="INIT")
    sw_w = W.Text(value="" if START_WEIGHTS is None else str(START_WEIGHTS), description="ckpt path",
                  placeholder="/kaggle/input/.../best_ckpt-3.pth")
    url_w = W.Text(value=DATASET_URL, description="data URL")
    ep_w = W.IntText(value=MAX_EPOCH_OVERRIDE, description="max epoch 0=auto")
    btn = W.Button(description="Apply settings", button_style="success")
    out = W.Output()

    def _apply(_=None):
        global STAGE, INIT, START_WEIGHTS, DATASET_URL, MAX_EPOCH_OVERRIDE
        STAGE = stage_w.value
        INIT = init_w.value
        START_WEIGHTS = Path(sw_w.value.strip()) if sw_w.value.strip() else None
        DATASET_URL = url_w.value.strip()
        MAX_EPOCH_OVERRIDE = int(ep_w.value)
        with out:
            out.clear_output()
            print_plan()

    btn.on_click(_apply)
    display(W.VBox([
        W.HBox([stage_w, init_w]),
        sw_w, url_w, ep_w,
        btn, out,
    ]))
except Exception as exc:
    print("No interactive widgets (", type(exc).__name__, ") — edit STAGE/INIT in this cell and re-run it.")


## 1. Install YOLOX


In [ ]:
def sh(cmd: str, stream: bool = True):
    print("+", cmd, flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    if not stream:
        subprocess.check_call(cmd, shell=True, env=env)
        return
    proc = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

if CLEAN_START and (YOLOX_DIR / ".git").is_dir():
    os.chdir(YOLOX_DIR)
    sh(f"git fetch origin {GIT_BRANCH}")
    sh(f"git checkout {GIT_BRANCH}")
    sh(f"git reset --hard origin/{GIT_BRANCH}")
elif not (YOLOX_DIR / "tools" / "train.py").exists():
    sh(f"git clone --depth 1 --branch {GIT_BRANCH} {REPO_URL} {YOLOX_DIR}")
os.chdir(YOLOX_DIR)
import torch
print(f"torch {torch.__version__} cuda={torch.cuda.is_available()}")
if STAGE != "export":
    assert torch.cuda.is_available(), "Turn on a GPU runtime."
sh(f"{sys.executable} -m pip install -q loguru thop ninja tabulate pycocotools tensorboard")
setup_py = YOLOX_DIR / "setup.py"
setup_py.write_text(setup_py.read_text().replace("ext_modules=get_ext_modules(),", "ext_modules=[],"))
sh(f"{sys.executable} -m pip install -e . --no-deps --no-build-isolation")
os.environ["PYTHONPATH"] = f"{YOLOX_DIR}:{os.environ.get('PYTHONPATH', '')}"
sys.path.insert(0, str(YOLOX_DIR))
import yolox
print("yolox", yolox.__file__)


## 2. Dataset


In [ ]:
import zipfile, urllib.request
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_yolo_root(root: Path) -> Path:
    if (root / "data.yaml").exists() or (root / "train" / "images").exists():
        return root
    for k in list(root.iterdir()) if root.exists() else []:
        if k.is_dir() and ((k / "data.yaml").exists() or (k / "train" / "images").exists()):
            return k
        if k.is_dir():
            for k2 in k.iterdir():
                if k2.is_dir() and ((k2 / "data.yaml").exists() or (k2 / "train" / "images").exists()):
                    return k2
    return root

YOLO_ROOT = None
if KAGGLE_DATA_DIR is not None:
    YOLO_ROOT = find_yolo_root(Path(KAGGLE_DATA_DIR))
elif IS_KAGGLE:
    inp = Path("/kaggle/input")
    hits = list(inp.rglob("data.yaml")) if inp.exists() else []
    if hits:
        YOLO_ROOT = find_yolo_root(hits[0].parent)
        print("Kaggle Input:", YOLO_ROOT)
if YOLO_ROOT is None:
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    zpath = ROOT / "dataset.zip"
    if DATASET_URL:
        urllib.request.urlretrieve(DATASET_URL, zpath)
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        DATASET_DIR.mkdir(parents=True)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(DATASET_DIR)
    elif DATASET_ZIP:
        with zipfile.ZipFile(DATASET_ZIP) as zf:
            zf.extractall(DATASET_DIR)
    YOLO_ROOT = find_yolo_root(DATASET_DIR)
print("YOLO_ROOT", YOLO_ROOT)
assert (YOLO_ROOT / "train" / "images").is_dir()
assert (YOLO_ROOT / "train" / "labels").is_dir()
assert (YOLO_ROOT / "valid" / "images").is_dir() or (YOLO_ROOT / "val" / "images").is_dir()
print("train images", len(list((YOLO_ROOT / "train" / "images").iterdir())))


## 3. Training recipe


In [ ]:
_sync_stage_knobs()
os.chdir(YOLOX_DIR)
_lr = (0.01 / 64.0)
if INIT == "continue":
    _lr *= FINETUNE_LR_SCALE
_warmup = 1 if INIT == "continue" else 5
val_split = "valid" if (YOLO_ROOT / "valid" / "images").is_dir() else "val"
data_dir = str(YOLO_ROOT.resolve())
exp_src = f"""#!/usr/bin/env python3
import os
import torch.nn as nn
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
from yolox_s_pallets import Exp as PalletExp

class Exp(PalletExp):
    def __init__(self):
        super().__init__()
        self.data_dir = {data_dir!r}
        self.train_split = "train"
        self.val_split = {val_split!r}
        self.num_classes = {NUM_CLASSES}
        self.depth = {CFG['depth']}
        self.width = {CFG['width']}
        self.input_size = ({IMG_SIZE}, {IMG_SIZE})
        self.test_size = ({IMG_SIZE}, {IMG_SIZE})
        self.max_epoch = {MAX_EPOCH}
        self.data_num_workers = {DATA_NUM_WORKERS}
        self.eval_interval = {EVAL_INTERVAL}
        self.no_aug_epochs = {NO_AUG_EPOCHS}
        self.save_history_ckpt = False
        self.basic_lr_per_img = {_lr}
        self.warmup_epochs = {_warmup}
        self.enable_mixup = {bool(CFG['mixup'])}
        self.mosaic_prob = {float(CFG['mosaic_prob'])}
        self.mosaic_scale = (0.5, 1.5)
        self.random_size = (10, 20)
        self.degrees = 30.0
        self.reg_weight = {REG_WEIGHT}
        self.angle_weight = {ANGLE_WEIGHT}
        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]

    def get_model(self):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03
        if getattr(self, "model", None) is None:
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act, depthwise={bool(CFG['depthwise'])})
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act, depthwise={bool(CFG['depthwise'])})
            self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        self.model.head.reg_weight = float(self.reg_weight)
        self.model.head.angle_weight = float(self.angle_weight)
        self.model.train()
        print(f"[finetune] freeze=full stage={STAGE} init={INIT}")
        return self.model
"""
KAGGLE_EXP = YOLOX_DIR / KAGGLE_EXP_REL
KAGGLE_EXP.parent.mkdir(parents=True, exist_ok=True)
KAGGLE_EXP.write_text(exp_src)
print("wrote", KAGGLE_EXP)


## 4. Resolve checkpoint (fresh vs continue)


In [ ]:
def find_uploaded_weight():
    if START_WEIGHTS is not None:
        p = Path(START_WEIGHTS)
        assert p.is_file(), f"START_WEIGHTS missing: {p}"
        return p
    inp = Path("/kaggle/input")
    if not inp.exists():
        return None
    hits = [p for p in inp.rglob("*") if p.is_file() and p.suffix.lower() in {".pth", ".pt"}]
    prefer = [p for p in hits if any(k in p.name.lower() for k in ("detect_best", "obb", "best_ckpt"))]
    if prefer:
        return prefer[0]
    return hits[0] if len(hits) == 1 else None


def resolve_start_ckpt():
    uploaded = find_uploaded_weight()
    if STAGE == "export":
        return None, "export uses bank files, not a start ckpt"
    if STAGE == "detect":
        if INIT == "fresh":
            return None, "public YOLOX-S (completely fresh detect)"
        for p, why in (
            (uploaded, "uploaded checkpoint"),
            (DETECT_BEST, "01_detect_best.pth in this session"),
            (BANK / "latest.pth", "latest.pth"),
        ):
            if p is not None and Path(p).is_file():
                return Path(p), "continue detect from " + why
        raise FileNotFoundError(
            "INIT=continue but no checkpoint found. Upload a .pth as Kaggle Input, "
            "or set INIT=fresh to start from public YOLOX-S."
        )
    # angle
    if INIT == "fresh":
        for p, why in (
            (DETECT_BEST, "01_detect_best.pth (fresh angle from detect)"),
            (uploaded, "uploaded detect checkpoint"),
        ):
            if p is not None and Path(p).is_file():
                return Path(p), why
        raise FileNotFoundError("Angle/fresh needs 01_detect_best.pth. Run detect first or upload it.")
    for p, why in (
        (ANGLE_BEST, "02_angle_best.pth (resume angle)"),
        (DETECT_BEST, "01_detect_best.pth"),
        (uploaded, "uploaded checkpoint"),
    ):
        if p is not None and Path(p).is_file():
            return Path(p), "continue angle from " + why
    raise FileNotFoundError("Angle/continue: no detect or angle checkpoint in the bank or Input.")


START_CKPT, START_WHY = resolve_start_ckpt()
print("START FILE:", START_CKPT if START_CKPT else "(download public YOLOX-S)")
print("BECAUSE:   ", START_WHY)
COCO_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if STAGE == "detect" and START_CKPT is None:
    if not COCO_WEIGHTS.exists():
        sh(f"wget -q --show-progress -O {COCO_WEIGHTS} {WEIGHTS_URL}")
    print("COCO", round(COCO_WEIGHTS.stat().st_size / 1e6, 1), "MB")
elif START_CKPT is not None:
    print("size", round(START_CKPT.stat().st_size / 1e6, 1), "MB")


## 5. Train (saves into pallet_weights/)


In [ ]:
if STAGE == "export":
    print("STAGE=export — skip train")
else:
    os.chdir(YOLOX_DIR)
    if WIPE_OUTPUTS and OUT_DIR.exists():
        shutil.rmtree(OUT_DIR)
        print("wiped trainer folder (fresh detect)")
    ckpt_arg = str(COCO_WEIGHTS.resolve()) if START_CKPT is None else str(Path(START_CKPT).resolve())
    assert Path(ckpt_arg).is_file(), ckpt_arg
    fp16_flag = "--fp16" if FP16 else ""
    print(f"TRAIN {STAGE}/{INIT} epochs={MAX_EPOCH} start={ckpt_arg}", flush=True)
    sh(f"{sys.executable} -u tools/train.py -f {KAGGLE_EXP_REL} -d {DEVICES} -b {BATCH} {fp16_flag} -c {ckpt_arg}")
    raw_best = OUT_DIR / "best_ckpt.pth"
    assert raw_best.is_file(), raw_best
    dest = DETECT_BEST if STAGE == "detect" else ANGLE_BEST
    shutil.copy2(raw_best, dest)
    shutil.copy2(raw_best, BANK / "latest.pth")
    print("SAVED", dest)


## 6. Score  (AP@0.50 = did we find it. Tiny val sets are noisy.)


In [ ]:
def latest_trained():
    for p in (ANGLE_BEST, DETECT_BEST, OUT_DIR / "best_ckpt.pth"):
        if p.is_file():
            return p
    raise FileNotFoundError("No trained weights yet")

BEST = latest_trained()
print("evaluating", BEST)
os.chdir(YOLOX_DIR)
fp16_flag = "--fp16" if FP16 and torch.cuda.is_available() else ""
sh(f"{sys.executable} -u tools/eval.py -f {KAGGLE_EXP_REL} -c {BEST} -d 1 -b {BATCH} --conf {EVAL_CONF} --nms {EVAL_NMS} {fp16_flag} --fuse")


## 7. Pictures (check tilt on **train** previews, not just val)


In [ ]:
from IPython.display import Image, display
import cv2
import torch as _torch
os.chdir(YOLOX_DIR)
val_img_dir = YOLO_ROOT / "valid" / "images"
if not val_img_dir.is_dir():
    val_img_dir = YOLO_ROOT / "val" / "images"
from yolox.data.datasets.yolov8_obb import _load_class_names
from yolox.exp import get_exp
from yolox.tools.demo import Predictor
from yolox.utils import fuse_model, get_model_info
exp = get_exp(str(KAGGLE_EXP_REL), None)
exp.test_conf, exp.nmsthre, exp.test_size = EVAL_CONF, EVAL_NMS, (IMG_SIZE, IMG_SIZE)
model = exp.get_model()
model.eval()
device = "gpu" if _torch.cuda.is_available() else "cpu"
if device == "gpu":
    model.cuda()
try:
    ckpt = _torch.load(BEST, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = _torch.load(BEST, map_location="cpu")
model.load_state_dict(ckpt["model"])
model = fuse_model(model)
print(get_model_info(model, exp.test_size))
predictor = Predictor(model, exp, _load_class_names(str(YOLO_ROOT)), device=device)
vis_dir = EXPORT / "previews"
vis_dir.mkdir(parents=True, exist_ok=True)
val_imgs = sorted(p for p in val_img_dir.iterdir() if p.suffix.lower() in IMG_EXTS)[:4]
train_imgs = sorted(p for p in (YOLO_ROOT / "train" / "images").iterdir() if p.suffix.lower() in IMG_EXTS)
step = max(1, len(train_imgs) // 6)
for p in list(val_imgs) + train_imgs[::step][:6]:
    outputs, img_info = predictor.inference(str(p))
    if outputs[0] is None:
        print(p.name, "NO BOX")
        continue
    out_p = vis_dir / f"{p.parent.parent.name}_{p.name}"
    cv2.imwrite(str(out_p), predictor.visual(outputs[0], img_info, predictor.confthre))
    display(Image(filename=str(out_p), width=480))


## 8. Export / download


In [ ]:
from IPython.display import FileLink, display
import torch as _torch
src = label = None
for p, name in ((ANGLE_BEST, "angle"), (DETECT_BEST, "detect"), (OUT_DIR / "best_ckpt.pth", "trainer"), (find_uploaded_weight(), "uploaded")):
    if p is not None and Path(p).is_file():
        src, label = Path(p), name
        break
assert src is not None, "Nothing to export — run detect first"
try:
    ckpt = _torch.load(src, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = _torch.load(src, map_location="cpu")
payload = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
EXPORT.mkdir(parents=True, exist_ok=True)
pt_path = EXPORT / f"yolox_{MODEL}_obb.pt"
_torch.save({"model": payload, "arch": f"yolox-{MODEL}-obb", "source": str(src), "stage": label}, pt_path)
shutil.copy2(src, EXPORT / src.name)
(EXPORT / "README.txt").write_text(
    f"YOLOX OBB weights from {src} ({label})\n"
    f"demo: python tools/demo.py webcam -f exps/example/custom/yolox_s_pallets.py -c yolox_{MODEL}_obb.pt --conf 0.25 --nms 0.45 --tsize 416\n"
)
print(pt_path, round(pt_path.stat().st_size / 1e6, 2), "MB from", src)
if IS_KAGGLE:
    display(FileLink(str(pt_path)))
zip_path = ROOT / "pallet_weights_export.zip"
if zip_path.exists():
    zip_path.unlink()
sh(f"cd {ROOT} && zip -r pallet_weights_export.zip pallet_weights export", stream=False)
print("zip", zip_path)
if IS_KAGGLE:
    display(FileLink(str(zip_path)))
